In [3]:
import pandas as pd

# 1. Load the generated dataset
df = pd.read_csv('iiko_mock_sales.csv')

# 2. Display the first 5 rows (Quick check)
display(df.head())

# 3. Calculate basic business metrics
total_revenue = df['Net_Revenue'].sum()
total_cogs = df['Total_COGS'].sum()
food_cost_pct = (total_cogs / total_revenue) * 100

print(f"Total Net Revenue: {total_revenue:,.0f} KZT")
print(f"Overall Food Cost: {food_cost_pct:.2f}%")

,Transaction_ID,Date,Waiter,Item,Category,Quantity,Total_COGS,Net_Revenue,Discount_Pct
0,TXN-10000,2026-09-13 13:20:00,Alina,Ribeye Steak,Food,3,10500,27000.0,0.0
1,TXN-10001,2026-10-24 10:08:00,Yerzhan,Caesar Salad,Food,1,700,2200.0,0.0
2,TXN-10002,2026-09-06 13:22:00,Alina,Margherita Pizza,Food,1,800,2500.0,0.0
3,TXN-10003,2026-10-22 16:03:00,Alina,Ribeye Steak,Food,1,3500,9000.0,0.0
4,TXN-10004,2026-09-18 11:48:00,Dias,Craft Beer,Bar,1,400,1500.0,0.0


Total Net Revenue: 48,213,240 KZT
Overall Food Cost: 34.62%


In [1]:
import sqlite3
import pandas as pd

# 1. Загружаем данные (на случай, если вы открыли VS Code заново)
df = pd.read_csv('iiko_mock_sales.csv')

# 2. Создаем виртуальную базу данных SQLite прямо в оперативной памяти
conn = sqlite3.connect(':memory:')

# 3. Отправляем нашу таблицу в SQL-базу под названием 'sales'
df.to_sql('sales', conn, index=False)

# 4. Пишем классический SQL-запрос для анализа работы официантов
query = """
SELECT 
    Waiter,
    COUNT(Transaction_ID) AS Total_Orders,
    SUM(Net_Revenue) AS Total_Revenue,
    ROUND(AVG(Net_Revenue), 0) AS Average_Check,
    SUM(CASE WHEN Discount_Pct > 0 THEN 1 ELSE 0 END) AS Discounted_Orders
FROM sales
GROUP BY Waiter
ORDER BY Total_Revenue DESC;
"""

# 5. Выполняем SQL-запрос и выводим результат
waiter_metrics = pd.read_sql_query(query, conn)
display(waiter_metrics)

,Waiter,Total_Orders,Total_Revenue,Average_Check,Discounted_Orders
0,Dias,2058,9909615.0,4815.0,790
1,Yerzhan,1986,9745590.0,4907.0,788
2,Ruslan,2007,9694685.0,4830.0,811
3,Madina,2000,9599085.0,4800.0,784
4,Alina,1949,9264265.0,4753.0,774


In [4]:
# 4. Analyze Food Cost by Category
print("--- Metrics by Category ---")
category_metrics = df.groupby('Category').agg(
    Total_Revenue=('Net_Revenue', 'sum'),
    Total_COGS=('Total_COGS', 'sum')
)
# Recalculate Food Cost % for each category
category_metrics['Food_Cost_Pct'] = (category_metrics['Total_COGS'] / category_metrics['Total_Revenue']) * 100

# Display sorted by highest Food Cost
display(category_metrics.sort_values(by='Food_Cost_Pct', ascending=False).round(2))


# 5. Analyze the impact of Discounts on Food Cost
print("\n--- Impact of Discounts ---")
discount_metrics = df.groupby('Discount_Pct').agg(
    Total_Revenue=('Net_Revenue', 'sum'),
    Total_COGS=('Total_COGS', 'sum'),
    Transactions=('Transaction_ID', 'count')
)
discount_metrics['Food_Cost_Pct'] = (discount_metrics['Total_COGS'] / discount_metrics['Total_Revenue']) * 100

display(discount_metrics.round(2))

--- Metrics by Category ---


,Total_Revenue,Total_COGS,Food_Cost_Pct
Category,,,
Food,35038010.0,13485900,38.49
Bar,10189750.0,2682400,26.32
Beverage,2985480.0,523800,17.54



--- Impact of Discounts ---


,Total_Revenue,Total_COGS,Transactions,Food_Cost_Pct
Discount_Pct,,,,
0.00,30494800.0,10025700,6053,32.88
0.10,8860590.0,3227600,1958,36.43
0.15,8857850.0,3438800,1989,38.82


In [ ]:
# 6. Calculate Gross Revenue (Revenue before discounts)
df['Gross_Revenue'] = df['Net_Revenue'] / (1 - df['Discount_Pct'])

# 7. Calculate Totals for comparison
total_gross_revenue = df['Gross_Revenue'].sum()
total_net_revenue = df['Net_Revenue'].sum()
total_cogs = df['Total_COGS'].sum()

# 8. Calculate Food Cost % for both scenarios
fc_without_discount = (total_cogs / total_gross_revenue) * 100
fc_with_discount = (total_cogs / total_net_revenue) * 100

# Calculate how much money was given away as discounts
lost_revenue = total_gross_revenue - total_net_revenue

# 9. Display the Executive Summary
print("--- 💰 Overall Profitability Impact ---")
print(f"Total COGS (Себестоимость):        {total_cogs:,.0f} KZT")
print(f"Gross Revenue (Без скидок):        {total_gross_revenue:,.0f} KZT")
print(f"Net Revenue (Фактическая с кассы): {total_net_revenue:,.0f} KZT")
print(f"Lost to Discounts (Сумма скидок):  {lost_revenue:,.0f} KZT\n")

print(f"✅ Ideal Food Cost (Без скидок):   {fc_without_discount:.2f}%")
print(f"❌ Actual Food Cost (Со скидками): {fc_with_discount:.2f}%")
print(f"📉 Margin lost due to discounts:   {fc_with_discount - fc_without_discount:.2f}%")

--- 💰 Overall Profitability Impact ---
Total COGS (Себестоимость):        16,692,100 KZT
Gross Revenue (Без скидок):        50,760,900 KZT
Net Revenue (Фактическая с кассы): 48,213,240 KZT
Lost to Discounts (Сумма скидок):  2,547,660 KZT

✅ Ideal Food Cost (Без скидок):   32.88%
❌ Actual Food Cost (Со скидками): 34.62%
📉 Margin lost due to discounts:   1.74%


In [5]:
# 6. Глубокий анализ чистой выручки и влияния скидок по официантам
query_detailed = """
SELECT 
    Waiter,
    COUNT(Transaction_ID) AS Total_Orders,
    ROUND(SUM(Net_Revenue / (1 - Discount_Pct)), 0) AS Gross_Revenue,
    SUM(Net_Revenue) AS Net_Revenue_Clean,
    ROUND(SUM(Net_Revenue / (1 - Discount_Pct)) - SUM(Net_Revenue), 0) AS Lost_To_Discounts,
    ROUND(AVG(Net_Revenue), 0) AS Average_Check,
    ROUND((CAST(SUM(CASE WHEN Discount_Pct > 0 THEN 1 ELSE 0 END) AS FLOAT) / COUNT(Transaction_ID)) * 100, 1) AS Discount_Freq_Pct
FROM sales
GROUP BY Waiter
ORDER BY Net_Revenue_Clean DESC;
"""

# Выполняем запрос и выводим результат напрямую
waiter_detailed = pd.read_sql_query(query_detailed, conn)
display(waiter_detailed)

,Waiter,Total_Orders,Gross_Revenue,Net_Revenue_Clean,Lost_To_Discounts,Average_Check,Discount_Freq_Pct
0,Dias,2058,10416700.0,9909615.0,507085.0,4815.0,38.4
1,Yerzhan,1986,10266400.0,9745590.0,520810.0,4907.0,39.7
2,Ruslan,2007,10213600.0,9694685.0,518915.0,4830.0,40.4
3,Madina,2000,10110400.0,9599085.0,511315.0,4800.0,39.2
4,Alina,1949,9753800.0,9264265.0,489535.0,4753.0,39.7


In [6]:
# 7. Расследование: в какие часы Руслан проводит чеки со скидками?
query_investigation = """
SELECT 
    strftime('%H', Date) AS Hour_of_Day,
    COUNT(Transaction_ID) AS Total_Discounted_Orders,
    ROUND(SUM(Net_Revenue / (1 - Discount_Pct)) - SUM(Net_Revenue), 0) AS Money_Lost
FROM sales
WHERE Waiter = 'Ruslan' AND Discount_Pct > 0
GROUP BY Hour_of_Day
ORDER BY Money_Lost DESC;
"""

# Выполняем запрос
ruslan_fraud_check = pd.read_sql_query(query_investigation, conn)

print("--- 🕵️‍♂️ Аудит скидок: Руслан (По часам) ---")
display(ruslan_fraud_check)

--- 🕵️‍♂️ Аудит скидок: Руслан (По часам) ---


,Hour_of_Day,Total_Discounted_Orders,Money_Lost
0,14,60,48635.0
1,19,62,48430.0
2,10,65,45475.0
3,15,61,44050.0
4,17,60,40160.0
5,20,64,40085.0
6,11,64,37485.0
7,13,57,36790.0
8,21,62,32780.0
9,23,52,32565.0
